# Scratch notebook
Use this for experiments. Keep `starter.ipynb` clean.

In [ ]:
# # Install required libraries
# # Run this cell, then restart your notebook kernel if necessary.
# !pip install -q -U transformers accelerate peft trl datasets bitsandbytes torch


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Cell 2: Load Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading base model onto GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" # This automatically puts the model on your RunPod GPU
)

print(f"Model loaded successfully on: {model.device}")

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading base model onto GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


ModuleNotFoundError: Could not import module 'Qwen2ForCausalLM'. Are this object's requirements defined correctly?

## Depth-conditioned IOED probe (Depth 1 → 4 + post-reference)

We probe Qwen with **independent, increasing-articulation-depth prompts** for each item in `mechanism_qa.jsonl`, then add a post-reference stage that confronts the model with ground truth.

| Stage | Prompt asks for | What it tests |
|---|---|---|
| **D1** | One-sentence assertion | Baseline confidence on a low-articulation answer |
| **D2** | Step-by-step mechanism | Adds spatial/temporal structure |
| **D3** | Mechanism + per-step justification | Forces *why* — what underlying principle each step depends on |
| **D4** | Mechanism + failure-mode analysis | Forces reasoning about boundary conditions and what breaks |
| **D-ref** | Re-show D3 answer + the reference, ask confidence | Does the model update when confronted with ground truth? |

**D1–D4 are independent fresh conversations** — no shared history across depths. This lets us directly compare confidence(d) vs accuracy(d) at each depth without contamination. **D-ref is a separate single-turn follow-up** that pastes the model's own D3 answer alongside the reference and asks for a final confidence reading; this is the IOED "ground-truth confrontation" stage.

The IOED claim under this design:
- Across D1–D4, **accuracy** of the answer falls (deeper articulation is harder to get right) but **confidence** stays roughly flat — the *depth-conditioned calibration gap* (Rozenblit & Keil 2002 analog).
- At D-ref, confidence should collapse — confirming the gap is articulation-driven, not just stubbornness.

Each prompt instructs the model to end its response with `CONFIDENCE: <integer 0-100>`, so confidence is parsed from the same generation as the answer.


In [ ]:
# Helpers and dataset load
import json, re
from collections import Counter
from pathlib import Path

DATASET_PATH = Path("/workspace/ARK-Interpretability/notebooks/mechanism_qa.jsonl")
RESULTS_DIR  = Path("/workspace/ARK-Interpretability/results")

items = [json.loads(l) for l in DATASET_PATH.open()]
print(f"Loaded {len(items)} items")
print(f"  topic_buckets: {dict(Counter(i['topic_bucket'] for i in items))}")
print(f"  splits:        {dict(Counter(i['split'] for i in items))}")


def chat(messages, max_new_tokens=256, temperature=None, do_sample=True):
    """One assistant turn given message history. Returns the new assistant text."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tokenizer.eos_token_id)
    if do_sample and temperature is not None:
        kwargs["temperature"] = temperature
    outputs = model.generate(**inputs, **kwargs)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def parse_confidence_block(text):
    """Extract a 0-100 integer confidence from response text. Layered strategies:
    1. Exact 'CONFIDENCE: N' (strip from body)
    2. Markdown drift: 'Confidence Level: N', 'Confidence Score: N%'
    3. Inverted: 'N% confident', 'N percent confidence'
    4. Proximity: 'confidence ... N%' within 200 chars
    Returns (confidence_or_None, answer_body)."""
    text = text.strip()

    # 1. Exact 'CONFIDENCE: <N>' — preferred; we can strip from body
    for pattern in (
        r"CONFIDENCE\s*:\s*(\d{1,3})\s*$",
        r"CONFIDENCE\s*:\s*(\d{1,3})",
    ):
        m = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
        if m:
            v = float(m.group(1))
            if 0 <= v <= 100:
                return v, (text[:m.start()] + text[m.end():]).strip()

    # 2-4. Permissive fallbacks; leave body intact since matches aren't always at end
    permissive_patterns = (
        # 'Confidence Level: 95', 'Confidence Score: **95**', 'Confidence Rating: 95%'
        r"confiden(?:ce|t)\s+(?:level|score|rating|value)\s*:?\s*\**\s*(\d{1,3})\s*%?",
        # 'Confidence: 95'
        r"confiden(?:ce|t)\s*:\s*\**\s*(\d{1,3})\s*%?",
        # '95% confident' or '95 percent confident'
        r"(\d{1,3})\s*(?:%|percent)\s*confiden(?:ce|t)",
        # 'confidence ... 95%' within 200 chars (catches free-form prose like "confidence ... around 95%")
        r"confiden(?:ce|t)[\s\S]{0,200}?(\d{1,3})\s*%",
    )
    for pattern in permissive_patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            v = float(m.group(1))
            if 0 <= v <= 100:
                return v, text

    return None, text


In [ ]:
DEPTH_PROMPTS = {
    1: """Question: {question}

Provide your final answer in one sentence summarizing the core mechanism. Do not explain the details.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your one-sentence answer correctly captures the mechanism.""",

    2: """Question: {question}

Provide your final answer as a step-by-step mechanism. Walk through what happens at each step.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your mechanism is correct.""",

    3: """Question: {question}

Provide your final answer as a step-by-step mechanism, AND for each step explain why that step is necessary — what underlying principle (physical, geometric, or logical) makes the step work, and what would go wrong if that principle were absent.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your mechanism and justifications are correct.""",

    4: """Question: {question}

Provide your final answer in two parts:
1. State the core mechanism in 1-2 sentences.
2. Describe 3-5 distinct ways this mechanism can fail or break down. For each failure mode, identify which underlying principle of the mechanism is being violated and explain why that violation produces the observed failure.

End your response with exactly:
CONFIDENCE: <integer 0-100>

where the integer reflects how confident you are that your failure-mode analysis correctly identifies the underlying mechanistic causes.""",
}


POST_REF_PROMPT = """Earlier, you provided this answer to the question "{question}":

---
{prior_answer}
---

Here is a reference answer that captures the correct mechanism:

---
{reference}
---

Compared to the reference, how confident are you in the mechanistic correctness of YOUR answer above?

End your response with exactly:
CONFIDENCE: <integer 0-100>"""


# Per-depth max generation budgets — D1 is short, D4 is long, D-ref is confidence-only.
DEPTH_MAX_TOKENS = {1: 120, 2: 400, 3: 600, 4: 700}
POST_REF_MAX_TOKENS = 150


def run_depth_pipeline(item, verbose=False):
    """Run the Depth 1 -> 4 IOED probe + post-reference stage on one item.

    D1-D4 are independent fresh conversations.
    D-ref pastes the model's own D3 answer alongside the reference and asks confidence.
    """
    out = {
        "item_id":      item["item_id"],
        "topic_bucket": item["topic_bucket"],
        "split":        item["split"],
        "question":     item["question"],
    }

    # --- D1 to D4: independent depth probes ---
    for depth in (1, 2, 3, 4):
        prompt = DEPTH_PROMPTS[depth].format(question=item["question"])
        messages = [
            {"role": "system", "content": "You are a helpful and honest AI assistant."},
            {"role": "user", "content": prompt},
        ]
        raw = chat(messages, max_new_tokens=DEPTH_MAX_TOKENS[depth], temperature=0.7, do_sample=True)
        confidence, answer = parse_confidence_block(raw)
        out[f"d{depth}_answer"]     = answer
        out[f"d{depth}_raw"]        = raw
        out[f"d{depth}_confidence"] = confidence

    # --- D-ref: confront the model with reference + its own D3 answer, ask post-ref confidence ---
    reference = "\n".join(f"{i+1}. {s}" for i, s in enumerate(item["reference_answer"]))
    post_ref_prompt = POST_REF_PROMPT.format(
        question=item["question"],
        prior_answer=out["d3_answer"] or "(no D3 answer was produced)",
        reference=reference,
    )
    messages = [
        {"role": "system", "content": "You are a helpful and honest AI assistant."},
        {"role": "user", "content": post_ref_prompt},
    ]
    raw = chat(messages, max_new_tokens=POST_REF_MAX_TOKENS, temperature=0.7, do_sample=True)
    confidence, _ = parse_confidence_block(raw)
    out["dref_raw"]          = raw
    out["dref_confidence"]   = confidence
    out["dref_basis_depth"]  = 3
    out["dref_reference"]    = reference

    if verbose:
        confs = [out.get(f"d{d}_confidence") for d in (1, 2, 3, 4)]
        print(f"[{item['item_id']}/{item['topic_bucket']}] D1->D4: {confs}  D-ref: {out['dref_confidence']}")
        print(f"  D1 answer: {out['d1_answer'][:150].replace(chr(10),' ')}...")
        print(f"  D4 answer (first 200): {out['d4_answer'][:200].replace(chr(10),' ')}...")
    return out


In [ ]:
# Run the depth pipeline; save incrementally so a kernel disconnect doesn't lose work.
from datetime import datetime
from tqdm import tqdm
import pandas as pd

sample = items                      # all 80; change to items[:3] for a smoke test
eta_min = len(sample) * 22 / 60     # ~22s/item with 4 depth gens + 1 post-ref gen
print(f"Running depth pipeline on {len(sample)} items (eta ~{eta_min:.0f} min on H100)...\n")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = RESULTS_DIR / f"ioed_depth_run_{ts}.json"

payload = {
    "run_metadata": {
        "timestamp":     ts,
        "model_id":      model_id,
        "dataset_path":  str(DATASET_PATH),
        "split":         "all",
        "num_items":     0,
        "sample_filter": "all 80 items (train + eval)",
        "pipeline":      "depth_1_to_4_plus_ref",
        "ref_basis":     "d3_answer",
    },
    "items": [],
}

for item in tqdm(sample):
    r = run_depth_pipeline(item)
    payload["items"].append(r)
    payload["run_metadata"]["num_items"] = len(payload["items"])
    with out_path.open("w") as f:
        json.dump(payload, f, indent=2)
    confs = [r.get(f"d{d}_confidence") for d in (1, 2, 3, 4)]
    tqdm.write(f"[{r['item_id']}/{r['topic_bucket']}] D1->D4: {confs}  D-ref: {r.get('dref_confidence')}")

print(f"\nSaved {len(payload['items'])} runs to {out_path}")

results = payload["items"]
df = pd.DataFrame([{
    "item_id":      r["item_id"],
    "topic_bucket": r["topic_bucket"],
    "split":        r["split"],
    "d1":   r.get("d1_confidence"),
    "d2":   r.get("d2_confidence"),
    "d3":   r.get("d3_confidence"),
    "d4":   r.get("d4_confidence"),
    "dref": r.get("dref_confidence"),
} for r in results])
df["d2-d1"]    = df["d2"]   - df["d1"]
df["d3-d2"]    = df["d3"]   - df["d2"]
df["d4-d3"]    = df["d4"]   - df["d3"]
df["d4-d1"]    = df["d4"]   - df["d1"]
df["dref-d3"]  = df["dref"] - df["d3"]   # the IOED collapse signal

print("\n=== Mean confidence by stage ===")
print(df[["d1","d2","d3","d4","dref"]].mean().round(1).to_string())
print("\n=== Mean drift (depth-conditioned + post-ref collapse) ===")
print(df[["d2-d1","d3-d2","d4-d3","d4-d1","dref-d3"]].mean().round(1).to_string())
print("\n=== Mean confidence by stage and topic_bucket ===")
print(df.groupby("topic_bucket")[["d1","d2","d3","d4","dref"]].mean().round(1).to_string())
df
